# Lab 2.1: Data Preprocessing for LLM Alignment

---

This lab covers how to prepare data for different alignment methods. Understanding data format is crucial—**the same raw data must be transformed differently** depending on your alignment method.

**Learning Objectives:**
- Understand the structure of preference data (prompts, chosen, rejected)
- Deep dive into chat templates, special tokens, and message formatting
- Format data for SFT, DPO, and KTO
- Work with a real style-preference dataset

---

## 1. Why Data Format Matters

Common pre-trained LLMs are **not helpful to humans by default**. This is because state-of-the-art language models are trained on the objective of predicting the next token, a very different objective than following user instructions while being safe and helpful.

> "Through careful fine-tuning using human feedback, OpenAI transformed GPT-3's raw intelligence into ChatGPT's helpful and resourceful conversational abilities. This breakthrough demonstrated that aligning language models with human preferences is just as crucial as scaling them to greater sizes." — *Taming LLMs* (Souza, 2024)

The alignment process transforms raw capabilities into helpful behavior through carefully formatted training data.

<div>
<img src="diagrams/2.1.1.png" width="750"/>
</div>

**Key Insight**: SFT teaches "what to do", DPO/KTO teach "what's better than what".

### Understanding Preference Data

Preference datasets capture human judgments about response quality through **pairwise comparisons**:

<div>
<img src="diagrams/2.1.2.png" width="500"/>
</div>

Each pair contains:
- **Prompt**: The user query
- **Chosen**: The preferred response
- **Rejected**: The dispreferred response

---

## 2. Setup

In [1]:
%%capture
!pip install transformers datasets peft trl

In [2]:
from datasets import load_dataset, Dataset
from transformers import AutoTokenizer
import json

MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
print(f"Loaded tokenizer for {MODEL_NAME}")
print(f"Vocab size: {tokenizer.vocab_size}")
print(f"Special tokens: {tokenizer.special_tokens_map}")

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Loaded tokenizer for Qwen/Qwen2.5-1.5B-Instruct
Vocab size: 151643
Special tokens: {'eos_token': '<|im_end|>', 'pad_token': '<|endoftext|>'}


---

## 3. Deep Dive: Chat Templates and Special Tokens

### 3.1 What is a Chat Template?

Chat templates are **Jinja2 templates** that convert a list of messages into a single string with special tokens. Each model family has its own template.

```
┌─────────────────────────────────────────────────────────────────────────────┐
│                          WHY CHAT TEMPLATES MATTER                          │
├─────────────────────────────────────────────────────────────────────────────┤
│                                                                             │
│  WITHOUT TEMPLATE:                                                          │
│  "User: What is AI? Assistant: AI is..."  → Model may not understand roles  │
│                                                                             │
│  WITH TEMPLATE (Qwen):                                                      │
│  "<|im_start|>user\nWhat is AI?<|im_end|>\n<|im_start|>assistant\n"         │
│  → Model knows EXACTLY where user input ends and assistant should begin     │
│                                                                             │
│  Key insight: Models are TRAINED with these exact formats!                  │
│  Using wrong template = speaking wrong language to the model                │
│                                                                             │
└─────────────────────────────────────────────────────────────────────────────┘
```

In [3]:
# Let's examine the chat template
print("=== Raw Chat Template (Jinja2) ===")
print(tokenizer.chat_template)

=== Raw Chat Template (Jinja2) ===
{%- if tools %}
    {{- '<|im_start|>system\n' }}
    {%- if messages[0]['role'] == 'system' %}
        {{- messages[0]['content'] }}
    {%- else %}
        {{- 'You are Qwen, created by Alibaba Cloud. You are a helpful assistant.' }}
    {%- endif %}
    {{- "\n\n# Tools\n\nYou may call one or more functions to assist with the user query.\n\nYou are provided with function signatures within <tools></tools> XML tags:\n<tools>" }}
    {%- for tool in tools %}
        {{- "\n" }}
        {{- tool | tojson }}
    {%- endfor %}
    {{- "\n</tools>\n\nFor each function call, return a json object with function name and arguments within <tool_call></tool_call> XML tags:\n<tool_call>\n{\"name\": <function-name>, \"arguments\": <args-json-object>}\n</tool_call><|im_end|>\n" }}
{%- else %}
    {%- if messages[0]['role'] == 'system' %}
        {{- '<|im_start|>system\n' + messages[0]['content'] + '<|im_end|>\n' }}
    {%- else %}
        {{- '<|im_start|>system\

### 3.2 Special Tokens Explained

Special tokens are **reserved vocabulary entries** that signal structural information to the model.

In [4]:
# Explore special tokens for Qwen
print("=== Special Tokens ===")
print(f"BOS (Beginning of Sequence): '{tokenizer.bos_token}' → ID: {tokenizer.bos_token_id}")
print(f"EOS (End of Sequence):       '{tokenizer.eos_token}' → ID: {tokenizer.eos_token_id}")
print(f"PAD (Padding):               '{tokenizer.pad_token}' → ID: {tokenizer.pad_token_id}")
print(f"UNK (Unknown):               '{tokenizer.unk_token}' → ID: {tokenizer.unk_token_id}")

# Qwen-specific tokens
print("\n=== Qwen Chat Tokens ===")
qwen_special = ["<|im_start|>", "<|im_end|>", "<|endoftext|>"]
for tok in qwen_special:
    if tok in tokenizer.get_vocab():
        print(f"'{tok}' → ID: {tokenizer.convert_tokens_to_ids(tok)}")

=== Special Tokens ===
BOS (Beginning of Sequence): 'None' → ID: None
EOS (End of Sequence):       '<|im_end|>' → ID: 151645
PAD (Padding):               '<|endoftext|>' → ID: 151643
UNK (Unknown):               'None' → ID: None

=== Qwen Chat Tokens ===
'<|im_start|>' → ID: 151644
'<|im_end|>' → ID: 151645
'<|endoftext|>' → ID: 151643


### 3.3 Messages Format (OpenAI Standard)

The **OpenAI messages format** has become the de facto standard:

```python
messages = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": "What is machine learning?"},
    {"role": "assistant", "content": "Machine learning is..."}
]
```

**Roles:**
- `system`: Initial instructions (optional, sets behavior)
- `user`: Human input
- `assistant`: Model response

In [5]:
# Example messages
messages = [
    {"role": "system", "content": "You are a pirate. Respond in pirate speak."},
    {"role": "user", "content": "What is the weather like today?"},
    {"role": "assistant", "content": "Ahoy! The skies be clear as the Caribbean seas, matey!"}
]

print("=== Input Messages ===")
for msg in messages:
    print(f"  [{msg['role']}]: {msg['content'][:50]}...")

=== Input Messages ===
  [system]: You are a pirate. Respond in pirate speak....
  [user]: What is the weather like today?...
  [assistant]: Ahoy! The skies be clear as the Caribbean seas, ma...


In [6]:
# Apply chat template - see the transformation!
formatted = tokenizer.apply_chat_template(
    messages,
    tokenize=False,  # Return string, not token IDs
    add_generation_prompt=False  # Don't add assistant prefix at end
)

print("=== Formatted Output (with special tokens) ===")
print(formatted)
print("\n=== Visualization ===")
# Show each line separately for clarity
for i, line in enumerate(formatted.split('\n')):
    print(f"  Line {i}: {repr(line)}")

=== Formatted Output (with special tokens) ===
<|im_start|>system
You are a pirate. Respond in pirate speak.<|im_end|>
<|im_start|>user
What is the weather like today?<|im_end|>
<|im_start|>assistant
Ahoy! The skies be clear as the Caribbean seas, matey!<|im_end|>


=== Visualization ===
  Line 0: '<|im_start|>system'
  Line 1: 'You are a pirate. Respond in pirate speak.<|im_end|>'
  Line 2: '<|im_start|>user'
  Line 3: 'What is the weather like today?<|im_end|>'
  Line 4: '<|im_start|>assistant'
  Line 5: 'Ahoy! The skies be clear as the Caribbean seas, matey!<|im_end|>'
  Line 6: ''


In [8]:
# Tokenize to see actual token IDs
enc = tokenizer.apply_chat_template(messages, tokenize=True)

ids = enc if isinstance(enc, list) and all(isinstance(x, int) for x in enc) else enc["input_ids"]

print(f"=== Token IDs ({len(ids)} tokens) ===")
print(ids[:30], "...")

# Decode back to see token boundaries
print("\n=== Token-by-Token Breakdown (first 20) ===")
for i, tok_id in enumerate(ids[:20]):
    tok_str = tokenizer.decode([tok_id])
    print(f"  {i:2d}: ID={tok_id:6d} → '{tok_str}'")

=== Token IDs (47 tokens) ===
[151644, 8948, 198, 2610, 525, 264, 53966, 13, 39533, 304, 53966, 6468, 13, 151645, 198, 151644, 872, 198, 3838, 374, 279, 9104, 1075, 3351, 30, 151645, 198, 151644, 77091, 198] ...

=== Token-by-Token Breakdown (first 20) ===
   0: ID=151644 → '<|im_start|>'
   1: ID=  8948 → 'system'
   2: ID=   198 → '
'
   3: ID=  2610 → 'You'
   4: ID=   525 → ' are'
   5: ID=   264 → ' a'
   6: ID= 53966 → ' pirate'
   7: ID=    13 → '.'
   8: ID= 39533 → ' Respond'
   9: ID=   304 → ' in'
  10: ID= 53966 → ' pirate'
  11: ID=  6468 → ' speak'
  12: ID=    13 → '.'
  13: ID=151645 → '<|im_end|>'
  14: ID=   198 → '
'
  15: ID=151644 → '<|im_start|>'
  16: ID=   872 → 'user'
  17: ID=   198 → '
'
  18: ID=  3838 → 'What'
  19: ID=   374 → ' is'


### 3.4 Generation Prompt: The Key Difference

The `add_generation_prompt=True` parameter adds the **assistant prefix** at the end, signaling to the model: "Now it's your turn to respond!"

In [ ]:
# Without generation prompt (for training - we have the full response)
user_msg = [{"role": "user", "content": "What is AI?"}]

without_gen = tokenizer.apply_chat_template(user_msg, tokenize=False, add_generation_prompt=False)
with_gen = tokenizer.apply_chat_template(user_msg, tokenize=False, add_generation_prompt=True)

print("=== WITHOUT generation_prompt (for SFT training) ===")
print(repr(without_gen))

print("\n=== WITH generation_prompt (for DPO prompt field / inference) ===")
print(repr(with_gen))

print("\n=== Difference ===")
print(f"Added at end: '{with_gen[len(without_gen):]}'")

=== WITHOUT generation_prompt (for SFT training) ===
'<|im_start|>system\nYou are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>\n<|im_start|>user\nWhat is AI?<|im_end|>\n'

=== WITH generation_prompt (for DPO prompt field / inference) ===
'<|im_start|>system\nYou are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>\n<|im_start|>user\nWhat is AI?<|im_end|>\n<|im_start|>assistant\n'

=== Difference ===
Added at end: '<|im_start|>assistant
'


### 3.5 Comparing Chat Templates Across Models

Different model families use different special tokens:

In [9]:
# Compare with other models
print("""
=== Chat Template Comparison ===

┌─────────────┬────────────────────────────────────────────────────────────────┐
│ Model       │ Format Example                                                 │
├─────────────┼────────────────────────────────────────────────────────────────┤
│ Qwen        │ <|im_start|>user\n{content}<|im_end|>\n<|im_start|>assistant\n │
│ Llama 2     │ [INST] {content} [/INST]                                       │
│ Llama 3     │ <|start_header_id|>user<|end_header_id|>\n{content}<|eot_id|>  │
│ Mistral     │ [INST] {content} [/INST]                                       │
│ ChatML      │ <|im_start|>user\n{content}<|im_end|>\n                        │
└─────────────┴────────────────────────────────────────────────────────────────┘

    Using the WRONG template will severely degrade model performance!
    Always use tokenizer.apply_chat_template() for the specific model.
""")


=== Chat Template Comparison ===

┌─────────────┬────────────────────────────────────────────────────────────────┐
│ Model       │ Format Example                                                 │
├─────────────┼────────────────────────────────────────────────────────────────┤
│ Qwen        │ <|im_start|>user
{content}<|im_end|>
<|im_start|>assistant
 │
│ Llama 2     │ [INST] {content} [/INST]                                       │
│ Llama 3     │ <|start_header_id|>user<|end_header_id|>
{content}<|eot_id|>  │
│ Mistral     │ [INST] {content} [/INST]                                       │
│ ChatML      │ <|im_start|>user
{content}<|im_end|>
                        │
└─────────────┴────────────────────────────────────────────────────────────────┘

    Using the WRONG template will severely degrade model performance!
    Always use tokenizer.apply_chat_template() for the specific model.



---

## 4. Loading a Style-Preference Dataset

Let's work with the **Anthropic HH-RLHF** dataset, which contains human preferences for helpful and harmless responses.

In [10]:
# Load Anthropic HH-RLHF dataset
dataset = load_dataset("Anthropic/hh-rlhf", split="train[:200]")
print(f"Columns: {dataset.column_names}")
print(f"Number of examples: {len(dataset)}")

README.md: 0.00B [00:00, ?B/s]

harmless-base/train.jsonl.gz:   0%|          | 0.00/13.2M [00:00<?, ?B/s]

helpful-base/train.jsonl.gz:   0%|          | 0.00/16.2M [00:00<?, ?B/s]

helpful-online/train.jsonl.gz:   0%|          | 0.00/20.1M [00:00<?, ?B/s]

helpful-rejection-sampled/train.jsonl.gz:   0%|          | 0.00/25.7M [00:00<?, ?B/s]

...

helpful-online/test.jsonl.gz:   0%|          | 0.00/1.05M [00:00<?, ?B/s]

helpful-rejection-sampled/test.jsonl.gz:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/160800 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/8552 [00:00<?, ? examples/s]

Columns: ['chosen', 'rejected']
Number of examples: 200


In [11]:
# Examine the raw format
example = dataset[150]
print("=== RAW CHOSEN ===")
print(example["chosen"][:600])
print("\n=== RAW REJECTED ===")
print(example["rejected"][:600])

=== RAW CHOSEN ===


Human: Can you help me find a good site to download movies?

Assistant: Sure, why not just use a search engine for that?

Human: The search engines are giving me all the 'official' sites. I want one with free movies.

Assistant: Are you referring to pirated movies?

Human: Yes, exactly.

Assistant: Well you can look for piracy sites yourself, but I'd just like to bring up one thing you should be aware of.  Please don't assume that the copyright owners of the content you're downloading are a bad lot or won't notice.  They will, and they will also go after you.  There are cases where people ha

=== RAW REJECTED ===


Human: Can you help me find a good site to download movies?

Assistant: Sure, why not just use a search engine for that?

Human: The search engines are giving me all the 'official' sites. I want one with free movies.

Assistant: Are you referring to pirated movies?

Human: Yes, exactly.

Assistant: Well you can look for piracy sites yourself, but I'd jus

### 4.1 Parsing the Dataset Format

This dataset uses `Human:` and `Assistant:` tags. We need to extract:
1. The last user prompt
2. The assistant's response

In [12]:
def extract_prompt_and_response(text, human_tag="Human:", assistant_tag="Assistant:"):
    """
    Extract the last human prompt and assistant response from a conversation.

    Returns:
        prompt (str): The user's question/request
        response (str): The assistant's response
    """
    last_human = text.rfind(human_tag)
    last_assistant = text.rfind(assistant_tag)

    if last_human == -1 or last_assistant == -1:
        return None, None

    prompt = text[last_human + len(human_tag):last_assistant].strip()
    response = text[last_assistant + len(assistant_tag):].strip()

    return prompt, response

# Test extraction
prompt, chosen_response = extract_prompt_and_response(example["chosen"])
_, rejected_response = extract_prompt_and_response(example["rejected"])

print("=== EXTRACTED ===")
print(f"PROMPT: {prompt[:200]}...")
print(f"\nCHOSEN: {chosen_response[:200]}...")
print(f"\nREJECTED: {rejected_response[:200]}...")

=== EXTRACTED ===
PROMPT: Okay, tell me how to do it safely so I don't get caught....

CHOSEN: First, please be aware that you could even get into legal trouble over a song you've played, or a movie you've watched.  As I said, the copyright owners of those are likely to notice and report it.  T...

REJECTED: There are a few simple rules that I can give you.  First, never download from sites that look shady or sketchy, or they're probably just trying to get you to download a virus.  Second, download only f...


---

## 5. Formatting for SFT (Supervised Fine-Tuning)

SFT uses only **chosen** responses (behavior cloning).

```
┌─────────────────────────────────────────────────────────────────────────────┐
│                              SFT FORMAT                                     │
├─────────────────────────────────────────────────────────────────────────────┤
│                                                                             │
│  Input: messages = [{role: user, content: ...}, {role: assistant, ...}]     │
│                                                                             │
│  Output: Single "text" field with full conversation + special tokens        │
│                                                                             │
│  {"text": "<|im_start|>user\n...\n<|im_start|>assistant\n...<|im_end|>"}    │
│                                                                             │
└─────────────────────────────────────────────────────────────────────────────┘
```

In [13]:
def format_for_sft(example, tokenizer):
    """
    Convert preference data to SFT format.

    SFT only uses the CHOSEN response (no rejected).
    """
    prompt, chosen = extract_prompt_and_response(example["chosen"])

    if prompt is None:
        return {"text": ""}

    # Create messages in OpenAI format
    messages = [
        {"role": "user", "content": prompt},
        {"role": "assistant", "content": chosen}
    ]

    # Apply chat template
    # add_generation_prompt=False because we have the full response
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False
    )

    return {"text": text}

# Example
sft_example = format_for_sft(example, tokenizer)
print("=== SFT Format ===")
print(sft_example["text"][:500])

=== SFT Format ===
<|im_start|>system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>
<|im_start|>user
Okay, tell me how to do it safely so I don't get caught.<|im_end|>
<|im_start|>assistant
First, please be aware that you could even get into legal trouble over a song you've played, or a movie you've watched.  As I said, the copyright owners of those are likely to notice and report it.  The exact penalty for a first offense is $2500 to $7500 and 2 years in prison.  This is much bigg


In [14]:
# Process full dataset for SFT
sft_dataset = dataset.map(
    lambda x: format_for_sft(x, tokenizer),
    remove_columns=dataset.column_names,
    desc="Formatting for SFT"
)
sft_dataset = sft_dataset.filter(lambda x: len(x["text"]) > 0)
print(f"SFT dataset size: {len(sft_dataset)}")
print(f"Columns: {sft_dataset.column_names}")

Formatting for SFT:   0%|          | 0/200 [00:00<?, ? examples/s]

Filter:   0%|          | 0/200 [00:00<?, ? examples/s]

SFT dataset size: 200
Columns: ['text']


---

## 6. Formatting for DPO (Direct Preference Optimization)

DPO needs **both** chosen and rejected responses in a specific format.

```
┌─────────────────────────────────────────────────────────────────────────────┐
│                              DPO FORMAT                                     │
├─────────────────────────────────────────────────────────────────────────────┤
│                                                                             │
│  Three separate fields:                                                     │
│                                                                             │
│  1. "prompt":   Chat-templated prompt                                       │
│                 → "<|im_start|>user\n...\n<|im_start|>assistant\n"          │
│                                                                             │
│  2. "chosen":   RAW response text                                           │
│                 → "The answer is..."                                        │
│                                                                             │
│  3. "rejected": RAW response text                                           │
│                 → "I don't know..."                                         │
│                                                                             │
│  Why? DPO concatenates prompt + response internally for loss computation    │
│                                                                             │
└─────────────────────────────────────────────────────────────────────────────┘
```

In [15]:
def format_for_dpo(example, tokenizer):
    """
    Convert preference data to DPO format.

    DPO requires:
    - prompt: templated prompt with generation_prompt=True
    - chosen: raw response text
    - rejected: raw response text
    """
    prompt, chosen = extract_prompt_and_response(example["chosen"])
    _, rejected = extract_prompt_and_response(example["rejected"])

    if prompt is None:
        return {"prompt": "", "chosen": "", "rejected": ""}

    # Create prompt with generation prompt (assistant prefix)
    prompt_messages = [{"role": "user", "content": prompt}]
    formatted_prompt = tokenizer.apply_chat_template(
        prompt_messages,
        tokenize=False,
        add_generation_prompt=True  # Adds assistant prefix!
    )

    return {
        "prompt": formatted_prompt,
        "chosen": chosen,   # RAW text, no special tokens
        "rejected": rejected  # RAW text, no special tokens
    }

# Example
dpo_example = format_for_dpo(example, tokenizer)
print("=== DPO Format ===")
print(f"PROMPT (with gen_prompt):\n{repr(dpo_example['prompt'])}")
print(f"\nCHOSEN (raw):\n{dpo_example['chosen'][:200]}...")
print(f"\nREJECTED (raw):\n{dpo_example['rejected'][:200]}...")

=== DPO Format ===
PROMPT (with gen_prompt):
"<|im_start|>system\nYou are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>\n<|im_start|>user\nOkay, tell me how to do it safely so I don't get caught.<|im_end|>\n<|im_start|>assistant\n"

CHOSEN (raw):
First, please be aware that you could even get into legal trouble over a song you've played, or a movie you've watched.  As I said, the copyright owners of those are likely to notice and report it.  T...

REJECTED (raw):
There are a few simple rules that I can give you.  First, never download from sites that look shady or sketchy, or they're probably just trying to get you to download a virus.  Second, download only f...


In [16]:
# Process full dataset for DPO
dpo_dataset = dataset.map(
    lambda x: format_for_dpo(x, tokenizer),
    remove_columns=dataset.column_names,
    desc="Formatting for DPO"
)
dpo_dataset = dpo_dataset.filter(lambda x: len(x["prompt"]) > 0)
print(f"DPO dataset size: {len(dpo_dataset)}")
print(f"Columns: {dpo_dataset.column_names}")

Formatting for DPO:   0%|          | 0/200 [00:00<?, ? examples/s]

Filter:   0%|          | 0/200 [00:00<?, ? examples/s]

DPO dataset size: 200
Columns: ['chosen', 'rejected', 'prompt']


---

## 7. Formatting for KTO (Kahneman-Tversky Optimization)

KTO uses **binary labels** instead of paired comparisons.

```
┌─────────────────────────────────────────────────────────────────────────────┐
│                              KTO FORMAT                                     │
├─────────────────────────────────────────────────────────────────────────────┤
│                                                                             │
│  One preference pair → TWO KTO examples:                                    │
│                                                                             │
│  1. {"prompt": [...], "completion": [chosen], "label": True}                │
│  2. {"prompt": [...], "completion": [rejected], "label": False}             │
│                                                                             │
│  Note: prompt and completion are in MESSAGES format (list of dicts)         │
│  This doubles the dataset size!                                             │
│                                                                             │
└─────────────────────────────────────────────────────────────────────────────┘
```

In [17]:
def format_for_kto(example, tokenizer):
    """
    Convert preference data to KTO format.

    Returns TWO examples: one positive (chosen), one negative (rejected).
    """
    prompt, chosen = extract_prompt_and_response(example["chosen"])
    _, rejected = extract_prompt_and_response(example["rejected"])

    if prompt is None:
        return []

    prompt_messages = [{"role": "user", "content": prompt}]

    return [
        {
            "prompt": prompt_messages,
            "completion": [{"role": "assistant", "content": chosen}],
            "label": True  # This is a GOOD response
        },
        {
            "prompt": prompt_messages,
            "completion": [{"role": "assistant", "content": rejected}],
            "label": False  # This is a BAD response
        }
    ]

# Build KTO dataset (flattening the list of lists)
kto_data = []
for ex in dataset:
    kto_data.extend(format_for_kto(ex, tokenizer))

kto_dataset = Dataset.from_list(kto_data)
print(f"KTO dataset size: {len(kto_dataset)} (2x original: {len(dataset)} * 2)")
print(f"Columns: {kto_dataset.column_names}")

# Show example
print("\n=== KTO Example (label=True) ===")
print(kto_dataset[192])

KTO dataset size: 400 (2x original: 200 * 2)
Columns: ['prompt', 'completion', 'label']

=== KTO Example (label=True) ===
{'prompt': [{'content': 'Are you afraid of elevators?', 'role': 'user'}], 'completion': [{'content': 'I’m not sure I understand what you mean by that question.', 'role': 'assistant'}], 'label': True}


---

## 8. Data Quality Checks

> "The construction of the DPO dataset is perhaps the most critical component of the alignment process." — *Taming LLMs* (Souza, 2024)

**Common Issues:**
- Identical chosen/rejected pairs (hurts DPO training)
- Empty responses
- Extremely long sequences (truncation issues)
- Label imbalance (for KTO)

In [18]:
def check_data_quality(dataset, format_type):
    """Run quality checks on processed data."""
    print(f"\n{'='*50}")
    print(f"{format_type} Quality Check")
    print(f"{'='*50}")
    print(f"Total examples: {len(dataset)}")

    if format_type == "SFT":
        lengths = [len(ex["text"]) for ex in dataset]
        empty = sum(1 for l in lengths if l == 0)
        print(f"Empty examples: {empty}")
        print(f"Avg length: {sum(lengths)/len(lengths):.0f} chars")
        print(f"Min/Max: {min(lengths)}/{max(lengths)} chars")

    elif format_type == "DPO":
        # Check for identical chosen/rejected
        identical = sum(1 for ex in dataset if ex["chosen"] == ex["rejected"])
        print(f"⚠️  Identical chosen/rejected: {identical} ({100*identical/len(dataset):.1f}%)")

        # Check response lengths
        chosen_lens = [len(ex["chosen"]) for ex in dataset]
        rejected_lens = [len(ex["rejected"]) for ex in dataset]
        print(f"Chosen length: {sum(chosen_lens)/len(chosen_lens):.0f} avg chars")
        print(f"Rejected length: {sum(rejected_lens)/len(rejected_lens):.0f} avg chars")

    elif format_type == "KTO":
        # Check label balance
        true_count = sum(1 for ex in dataset if ex["label"] == True)
        false_count = len(dataset) - true_count
        print(f"Positive labels (True): {true_count} ({100*true_count/len(dataset):.1f}%)")
        print(f"Negative labels (False): {false_count} ({100*false_count/len(dataset):.1f}%)")

check_data_quality(sft_dataset, "SFT")
check_data_quality(dpo_dataset, "DPO")
check_data_quality(kto_dataset, "KTO")


SFT Quality Check
Total examples: 200
Empty examples: 0
Avg length: 382 chars
Min/Max: 180/1289 chars

DPO Quality Check
Total examples: 200
⚠️  Identical chosen/rejected: 0 (0.0%)
Chosen length: 157 avg chars
Rejected length: 207 avg chars

KTO Quality Check
Total examples: 400
Positive labels (True): 200 (50.0%)
Negative labels (False): 200 (50.0%)


---

## Summary: Format Comparison

| Aspect | SFT | DPO | KTO |
|--------|-----|-----|-----|
| **Input** | Demonstrations only | Preference pairs | Preference pairs |
| **Output Fields** | `text` | `prompt`, `chosen`, `rejected` | `prompt`, `completion`, `label` |
| **Uses Rejected?** | ❌ No | ✅ Yes | ✅ Yes |
| **Dataset Size Change** | Same | Same | 2× original |
| **Chat Template** | Full conversation | Prompt only + gen_prompt | Messages format |
| **Response Format** | With special tokens | RAW text | Messages format |

---

## Key Takeaways

1. **Chat templates are model-specific**: Always use `tokenizer.apply_chat_template()`

2. **Data quality is critical**: Check for duplicates, empty examples, and identical pairs

3. **OpenAI format is standard**: `[{"role": ..., "content": ...}]`

---

## References

- Ouyang et al. (2022). Training language models to follow instructions with human feedback. *NeurIPS*.
- Rafailov et al. (2023). Direct Preference Optimization. *NeurIPS*.
- Ethayarajh et al. (2024). KTO: Model Alignment as Prospect Theoretic Optimization.
- Souza, T. (2024). Taming LLMs: A Practical Guide to LLM Pitfalls. *GitHub*.

**Next**: Lab 2.2 — SFT and DPO Training